In [52]:
from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator

In [68]:
backend = AerSimulator()
estimator = Estimator(backend)


def w_state_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(0, n - 1):
        qc.cx(i, i + 1)
    return qc


n = 4
qc = w_state_circuit(n)

In [69]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
transpiled_circuit = pm.run(qc)

In [70]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

# Pauli decompositions of m0..m3
m_decomp = {
    0: SparsePauliOp.from_list([("I", 0.5), ("Z", 0.5)]),
    3: SparsePauliOp.from_list([("I", 0.5), ("Z", -0.5)]),
    1: SparsePauliOp.from_list([("X", 0.5), ("Y", 0.5j)]),
    2: SparsePauliOp.from_list([("X", 0.5), ("Y", -0.5j)]),
}


def tensor_paulis(pauli_ops):
    out = pauli_ops[0]
    for p in pauli_ops[1:]:
        out = out.tensor(p)
    return out

rng = np.random.default_rng(seed=0)

M = 50

Oj_paulis = []          # non-Hermitian Pauli operators
Oj_indices = []         # for reproducibility / inspection

for _ in range(M):
    indices = rng.integers(0, 4, size=n)
    ops = [m_decomp[k] for k in indices]
    Oj = tensor_paulis(ops)
    Oj_paulis.append(Oj)
    Oj_indices.append(indices)


H_ops = []   # Hermitian part
K_ops = []   # skew-Hermitian part (made Hermitian)

for Oj in Oj_paulis:
    Oj_dag = Oj.adjoint()

    Hj = 0.5 * (Oj + Oj_dag)
    Kj = (Oj - Oj_dag) * (-0.5j)   # (O - O†)/(2i)

    H_ops.append(Hj)
    K_ops.append(Kj)


In [86]:
from qiskit.quantum_info import SparseObservable

y_real = []
y_imag = []

H_ops_kept = []
K_ops_kept = []

for Hj, Kj in zip(H_ops, K_ops):
    # Measure K_j only if nonzero:
    if SparseObservable.from_sparse_pauli_op(Kj).simplify() != SparseObservable.zero(n):
        # Always measure H_j
        res = estimator.run(
            [(transpiled_circuit, Hj)]
        ).result()
        y_real.append(res[0].data.evs)

        res = estimator.run(
            [(transpiled_circuit, Kj)]
        ).result()
        y_imag.append(res[0].data.evs)
        K_ops_kept.append(Kj)
        H_ops_kept.append(Hj)


In [72]:
training_data = {
    "H_ops": H_ops,        # Hermitian observables
    "K_ops": K_ops,
    "y_real": np.array(y_real),
    "y_imag": np.array(y_imag),
    "indices": Oj_indices # optional bookkeeping
}